
# 兰州气温时间序列综合分析

本脚本演示如何用 Matplotlib 的\ **面向对象接口**，把一套整年的逐日气温数据
绘制成一张 3 行子图的综合图板：上图为逐日平均温的年周期曲线（叠加月均温
并标注冬夏极值点），中图为日最高温与日最低温的相关性散点，下图为全年气温
分布直方图。

数据全部在脚本内用 NumPy 构造，无需访问任何外部文件（自包含）。我们用
年周期余弦叠加随机扰动来模拟兰州的"冬冷夏热"：峰值落在 7 月中旬，谷值
落在 1 月中旬，与真实气候规律一致。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ---- ① 导入 + 中文字体配置 ------------------------------------------
plt.rcParams["font.sans-serif"] = [
    "SimHei", "Microsoft YaHei", "Noto Sans CJK SC", "WenQuanYi Micro Hei",
]
plt.rcParams["axes.unicode_minus"] = False  # 让负号正常显示，而不是方块
plt.rcParams["figure.dpi"] = 120

---- ② 构造 2024 全年逐日最高/最低气温（自包含） -----------------------



In [ ]:
rng = np.random.default_rng(2024)

# 2024 年为闰年：366 天，各月天数用于后期算月平均
days_per_month = [31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
doy = np.arange(1, 367)                       # 一年中的第 1~366 天
month = np.repeat(np.arange(1, 13), days_per_month)

# 逐日平均温 = 年均温 + 年周期余弦（峰值第 202 天≈7 月下旬，谷值 1 月中旬）
#            + 随机扰动（模拟天气过程的日常波动）
day_mean = (10.5
            + 15.0 * np.cos(2 * np.pi * (doy - 202) / 365)
            + rng.normal(0, 1.8, 366))

# 昼夜温差：内陆城市温差大，约 6±2 ℃，再叠噪声
diurnal = 6.0 + rng.normal(0, 2.0, 366)
tmax = day_mean + diurnal                       # 逐日最高温
tmin = day_mean - diurnal                       # 逐日最低温

---- ③ 计算日平均（继续沿用）、月平均 ----------------------------------
日平均温 = 最高温与最低温的平均（比上面的 day_mean 又稳了一层）



In [ ]:
daily_avg = (tmax + tmin) / 2.0

# 每月平均温：把属于同一个月的气温取均值
monthly_mean = np.array([daily_avg[month == m].mean() for m in range(1, 13)])
# 每月撞在第几个月的中点（第几天），用于把月均温叠加到时间轴上
month_mid = np.array([np.where(month == m)[0].mean() for m in range(1, 13)])

# ---- 输出关键统计 -----------------------------------------------------
i_hot = int(np.argmax(daily_avg))
i_cold = int(np.argmin(daily_avg))
print(f"年度最热：第 {i_hot + 1} 天（7 月），{daily_avg[i_hot]:.1f} ℃")
print(f"年度最冷：第 {i_cold + 1} 天（1 月），{daily_avg[i_cold]:.1f} ℃")
print(f"月均温范围：{monthly_mean.min():.1f} ℃（{monthly_mean.argmin() + 1} 月） "
      f"~ {monthly_mean.max():.1f} ℃（{monthly_mean.argmax() + 1} 月）")
print(f"年均温：{daily_avg.mean():.1f} ℃")

---- ④ 绘图：3 行子图，依次为折线 / 散点 / 直方图 -----------------------



In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 14))

# [上] 全年逐日平均温曲线 + 月均温折线叠加 + 冬夏极值标注
axes[0].plot(doy, daily_avg, color="tab:blue", lw=1.2, label="逐日平均温")
axes[0].plot(month_mid, monthly_mean, color="tab:red", lw=2.2,
             marker="o", ms=5, label="月平均温")
axes[0].annotate(
    f"7 月最热 {daily_avg[i_hot]:.1f} ℃（第 {i_hot + 1} 天）",
    xy=(i_hot + 1, daily_avg[i_hot]),
    xytext=(i_hot - 70, daily_avg[i_hot] + 6),
    arrowprops=dict(arrowstyle="->", color="k"), fontsize=10)
axes[0].annotate(
    f"1 月最冷 {daily_avg[i_cold]:.1f} ℃（第 {i_cold + 1} 天）",
    xy=(i_cold + 1, daily_avg[i_cold]),
    xytext=(i_cold + 190, daily_avg[i_cold] - 8),
    arrowprops=dict(arrowstyle="->", color="k"), fontsize=10)
axes[0].set_title("① 兰州 2024 逐日平均温：冬冷夏热的年周期")
axes[0].set_ylabel("气温（℃）")
axes[0].set_xlim(0, 370)
axes[0].legend(loc="best")
axes[0].grid(ls="--", alpha=0.3)

# [中] 日最高温 vs 日最低温散点（看相关）+ 参考线 + 按月份着色
sc = axes[1].scatter(tmin, tmax, c=month, cmap="turbo", s=14, alpha=0.75,
                     edgecolors="none")
lo, hi = axes[1].get_xlim()
axes[1].plot([-20, 50], [-20, 50], ls="--", color="gray", lw=1.2,
             label="tmax = tmin（温差为 0）")
axes[1].set_title("② 日最高温 vs 日最低温散点（颜色 = 月份）")
axes[1].set_xlabel("日最低温（℃）")
axes[1].set_ylabel("日最高温（℃）")
axes[1].legend(loc="upper left")
axes[1].grid(ls="--", alpha=0.3)
fig.colorbar(sc, ax=axes[1], label="月份", location="right")

# [下] 全年气温分布直方图（看偏态 / 双峰 / 昼夜温差）
bins = np.arange(np.floor(tmin.min()), np.ceil(tmax.max()) + 1, 1.0)
axes[2].hist(daily_avg, bins=bins, color="tab:green", edgecolor="white",
             alpha=0.9, label="逐日平均温")
axes[2].axvline(daily_avg.mean(), color="k", ls="--", lw=1.5,
                label=f"年均温 {daily_avg.mean():.1f} ℃")
axes[2].set_title("③ 全年气温分布直方图（偏态 / 双峰一目了然）")
axes[2].set_xlabel("气温（℃）")
axes[2].set_ylabel("频数（天）")
axes[2].legend(loc="best")
axes[2].grid(ls="--", alpha=0.3)

fig.suptitle("兰州 2024 全年气温综合分析", fontsize=16)
fig.tight_layout()
plt.show()